In [1]:
import pandas as pd, numpy as np, torch, torchvision, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import sys
sys.path.insert(0, '../')
from util import *
from models.cnn import CNN

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)


# access each scene folder
scenarios = [f"../CSV_Files/scene{i+1}/" for i in range(6)]



device: NVIDIA A30


# Helper functions

In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    """Load one scenario, run preprocessing, return tensors + shapes."""
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def make_model(device, seed=0):
    """Fresh CNN with Xavier init, to match MATLAB's Glorot default."""
    torch.manual_seed(seed)
    model = CNN(n_classes=8).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    """MATLAB-style 'shuffle once' then fixed-order iteration."""
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    """Train + evaluate CNN on one scenario. Returns a dict of results."""
    # data
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    # model + optimizer
    model = make_model(device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ===")
        print(f"  shapes: train={tuple(X_tr.shape)} val={tuple(X_va.shape)} "
              f"test={tuple(X_te.shape)}")
        print(f"  train labels: {np.bincount(y_tr.numpy())}")

    # training loop
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                          criterion, optimizer, device)
        va_loss, va_acc = evaluate_loader(model, val_loader, criterion, device)
        scheduler.step()
        if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
            print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.3f}")

    # test evaluation
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")

    return {
        "scenario":  scenario_idx,
        "n_train":   len(X_tr),
        "accuracy":  acc,
        "precision": p,
        "recall":    r,
        "f1":        f,
        "confusion": cm,
        "model":     model,                        # keep if you want to inspect later
    }


In [4]:
results = []
for i, sc_dir in enumerate(scenarios, start=1):
    r = run_scenario(i, sc_dir, device, epochs=70, lr=1e-2, seed=0)
    results.append(r)



=== Scenario 1 ===
  shapes: train=(7858, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [ 858 1000 1000 1000 1000 1000 1000 1000]
  ep   1 | train loss 1.6700 acc 0.413 | val loss 1.4306 acc 0.517
  ep  10 | train loss 0.8027 acc 0.692 | val loss 0.8029 acc 0.691
  ep  20 | train loss 0.6796 acc 0.736 | val loss 0.6931 acc 0.751
  ep  30 | train loss 0.5741 acc 0.785 | val loss 0.5979 acc 0.812
  ep  40 | train loss 0.5443 acc 0.794 | val loss 0.5602 acc 0.830
  ep  50 | train loss 0.4968 acc 0.820 | val loss 0.5235 acc 0.859
  ep  60 | train loss 0.4812 acc 0.824 | val loss 0.5078 acc 0.863
  ep  70 | train loss 0.4587 acc 0.833 | val loss 0.5006 acc 0.861
  TEST: acc=0.8594  P=0.8672  R=0.8594  F1=0.8578

=== Scenario 2 ===
  shapes: train=(3939, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [439 500 500 500 500 500 500 500]
  ep   1 | train loss 1.8310 acc 0.353 | val loss 1.6506 acc 0.465
  ep  10 | train loss 0.9523 acc 0.651 | val loss 0.9606 acc 0.67

In [5]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "n_train", "accuracy", "precision",
                       "recall", "f1")}
    for r in results
])
summary[["accuracy", "precision", "recall", "f1"]] = (
    summary[["accuracy", "precision", "recall", "f1"]].round(4))
print("\n=== Summary across scenarios ===")
print(summary.to_string(index=False))



=== Summary across scenarios ===
 scenario  n_train  accuracy  precision  recall     f1
        1     7858    0.8594     0.8672  0.8594 0.8578
        2     3939    0.8262     0.8380  0.8262 0.8263
        3      786    0.6838     0.7138  0.6838 0.6835
        4      391    0.6025     0.6304  0.6025 0.5996
        5      238    0.5356     0.5867  0.5356 0.5455
        6       77    0.3700     0.3964  0.3700 0.3774


In [6]:
for r in results:
    print(f"\nScenario {r['scenario']}  (n_train={r['n_train']}, acc={r['accuracy']:.3f})")
    print(r["confusion"])



Scenario 1  (n_train=7858, acc=0.859)
[[144   6   0  42   0   0   7   1]
 [  0 185   0   5   0   1   0   9]
 [  0   0 197   0   0   1   0   2]
 [ 21   5   2 153   0   0  19   0]
 [  0   0   0   0 200   0   0   0]
 [  0  11   5   0   0 132   0  52]
 [  1   0   0  12   1   0 186   0]
 [  0  12   2   3   0   5   0 178]]

Scenario 2  (n_train=3939, acc=0.826)
[[147   4   0  39   0   2   7   1]
 [  2 149   0   6   0   5   1  37]
 [  0   0 193   0   0   2   0   5]
 [ 47   5   0 133   0   0  15   0]
 [  0   0   0   0 200   0   0   0]
 [  1  10   5   0   0 134   0  50]
 [  1   0   0  20   0   0 179   0]
 [  3   4   3   1   0   2   0 187]]

Scenario 3  (n_train=786, acc=0.684)
[[107   1   0  51   0   2  23  16]
 [ 26  97   5   6   0   9   0  57]
 [  1   0 172   0   1  10   1  15]
 [ 56   1   1 107   0   1  24  10]
 [  0   0   0   0 199   1   0   0]
 [  2   7  20   2   2 103   0  64]
 [  8   0   0  45   0   0 147   0]
 [  8   7  18   2   1   2   0 162]]

Scenario 4  (n_train=391, acc=0.603)
[[ 